# Bloque 1: Configuración e Importación de dependencias

# 🏐 Sistema de Tracking de Jugadores de Vóley Playa

Este notebook implementa un sistema completo de seguimiento y análisis de jugadores en partidos de vóley playa utilizando:
- **YOLO11n** para detección de personas
- **Homografía** para mapeo de coordenadas video → campo
- **Algoritmos de corrección de IDs** para tracking robusto

## Estructura del Pipeline:
1. **Configuración**: Importar librerías y cargar modelo
2. **POI (Points of Interest)**: Seleccionar puntos de referencia
3. **Detección y Tracking**: Procesar video con YOLO + corrección de IDs
4. **Exportación**: Guardar datos en CSV/JSON
5. **Visualización**: Generar video y trayectorias

In [1]:
import cv2
import numpy as np
from ultralytics import YOLO
from collections import defaultdict
import os
import csv
import json

## 📦 Importación de Librerías

**Qué hace:** Importa todas las librerías necesarias para el proyecto.

**Cómo lo hace:**
- `cv2`: Procesamiento de imágenes y video (OpenCV)
- `numpy`: Operaciones matriciales y cálculos numéricos
- `YOLO`: Modelo de detección de objetos (ultralytics)
- `defaultdict`: Estructuras de datos eficientes
- `os`, `csv`, `json`: Manejo de archivos y datos

In [10]:
VIDEO_PATH = "VideosAnalisis\\clip 2 ‐ Hecho con Clipchamp.mp4"
MAPA_PATH = "beachvolleyballcourt.png"
MODEL_PATH = "yolo11n.pt"
MARGIN_PERCENT = 0.10
EXPECTED_PLAYERS = 4

# Archivo JSON para guardar/cargar puntos del mapa
PUNTOS_MAPA_JSON = "Output/puntos_mapa.json"

# Crear directorio de salida si no existe
os.makedirs("Output", exist_ok=True)

print("✓ Configuración cargada")

model = YOLO(MODEL_PATH)
print("✓ Modelo YOLO cargado")


✓ Configuración cargada
✓ Modelo YOLO cargado


## ⚙️ Configuración de Parámetros y Carga del Modelo

**Qué hace:** Define las rutas de archivos, parámetros del sistema y carga el modelo YOLO.

**Cómo lo hace:**
- **VIDEO_PATH**: Ruta al video a analizar
- **MAPA_PATH**: Imagen del campo de vóley playa (vista superior)
- **MODEL_PATH**: Modelo YOLO11n preentrenado
- **MARGIN_PERCENT**: Margen de expansión del polígono (10%)
- **EXPECTED_PLAYERS**: Número esperado de jugadores (4)
- **PUNTOS_MAPA_JSON**: Archivo para guardar puntos de referencia
- Crea el directorio `Output/` si no existe
- Inicializa el modelo YOLO para detección de personas (clase 0)

## Definición de funciones auxiliares

In [3]:
def get_points(event, x, y, flags, params):
    """Callback para seleccionar puntos con el mouse."""
    points = params["points"]
    image = params["image"]
    wname = params["wname"]
    max_points = params["max_points"]
    if event == cv2.EVENT_LBUTTONDOWN and len(points) < max_points:
        points.append([x, y])
        cv2.circle(image, (x, y), 6, (0, 0, 255), -1)
        cv2.putText(image, str(len(points)), (x + 5, y - 5),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 0, 0), 2)
        cv2.imshow(wname, image)
        if len(points) == max_points:
            cv2.waitKey(300)
            cv2.destroyWindow(wname)

def point_in_polygon_with_margin(point, polygon, margin_percent=0.20):
    """Verifica si un punto está dentro de un polígono expandido uniformemente en todas direcciones."""
    center = np.mean(polygon, axis=0)
    expanded_polygon = []
    
    for pt in polygon:
        # Expandir desde el centro hacia afuera en todas direcciones
        direction = pt - center
        # Aplicar el mismo margen en todas direcciones (arriba, abajo, izquierda, derecha)
        expanded_pt = pt + direction * margin_percent
        expanded_polygon.append(expanded_pt)
    
    expanded_polygon = np.array(expanded_polygon, dtype=np.int32)
    result = cv2.pointPolygonTest(expanded_polygon, point, False)
    return result >= 0
def calculate_iou(box1, box2):
    """Calcula Intersection over Union entre dos bounding boxes."""
    x1_1, y1_1, x2_1, y2_1 = box1
    x1_2, y1_2, x2_2, y2_2 = box2
    
    x1_i = max(x1_1, x1_2)
    y1_i = max(y1_1, y1_2)
    x2_i = min(x2_1, x2_2)
    y2_i = min(y2_1, y2_2)
    
    if x2_i < x1_i or y2_i < y1_i:
        return 0.0
    
    intersection = (x2_i - x1_i) * (y2_i - y1_i)
    area1 = (x2_1 - x1_1) * (y2_1 - y1_1)
    area2 = (x2_2 - x1_2) * (y2_2 - y1_2)
    union = area1 + area2 - intersection
    
    return intersection / union if union > 0 else 0.0
def get_track_info(tracking_data, track_id):
    """Obtiene información completa de un track."""
    frames = []
    positions = []
    bboxes = []
    
    for frame_idx, detections in tracking_data.items():
        for det in detections:
            if det[0] == track_id:
                frames.append(frame_idx)
                positions.append((det[5], det[6]))
                bboxes.append((det[1], det[2], det[3], det[4]))
    
    if not frames:
        return None, None, [], []
    
    return min(frames), max(frames), positions, bboxes
def should_merge_ids(tracking_data, id1, id2, max_gap=5, max_distance=150, min_iou=0.3):
    """Determina si dos IDs deberían fusionarse."""
    first1, last1, positions1, bboxes1 = get_track_info(tracking_data, id1)
    first2, last2, positions2, bboxes2 = get_track_info(tracking_data, id2)
    
    if first1 is None or first2 is None:
        return False
    
    # CASO 1: Secuencial
    if last1 < first2:
        gap = first2 - last1
        if gap <= max_gap:
            last_pos1 = positions1[-1]
            first_pos2 = positions2[0]
            distance = np.sqrt((last_pos1[0] - first_pos2[0])**2 + 
                              (last_pos1[1] - first_pos2[1])**2)
            
            last_bbox1 = bboxes1[-1]
            first_bbox2 = bboxes2[0]
            size1 = (last_bbox1[2] - last_bbox1[0]) * (last_bbox1[3] - last_bbox1[1])
            size2 = (first_bbox2[2] - first_bbox2[0]) * (first_bbox2[3] - first_bbox2[1])
            size_ratio = min(size1, size2) / max(size1, size2) if max(size1, size2) > 0 else 0
            
            if distance <= max_distance and size_ratio > 0.5:
                return True
    
    # CASO 2: Solapamiento
    overlap_start = max(first1, first2)
    overlap_end = min(last1, last2)
    
    if overlap_start <= overlap_end:
        overlapping_frames = []
        for frame_idx in range(overlap_start, overlap_end + 1):
            if frame_idx not in tracking_data:
                continue
            
            bbox1, bbox2 = None, None
            pos1, pos2 = None, None
            
            for det in tracking_data[frame_idx]:
                if det[0] == id1:
                    bbox1 = (det[1], det[2], det[3], det[4])
                    pos1 = (det[5], det[6])
                if det[0] == id2:
                    bbox2 = (det[1], det[2], det[3], det[4])
                    pos2 = (det[5], det[6])
            
            if bbox1 and bbox2:
                distance = np.sqrt((pos1[0] - pos2[0])**2 + (pos1[1] - pos2[1])**2)
                iou = calculate_iou(bbox1, bbox2)
                overlapping_frames.append((distance, iou))
        
        if overlapping_frames:
            avg_distance = np.mean([d for d, _ in overlapping_frames])
            avg_iou = np.mean([iou for _, iou in overlapping_frames])
            
            if avg_iou >= min_iou or avg_distance <= max_distance * 0.5:
                return True
    
    return False
def merge_track_ids(tracking_data, id_from, id_to):
    """Fusiona id_from en id_to."""
    for frame_idx in tracking_data:
        new_detections = []
        for det in tracking_data[frame_idx]:
            if det[0] == id_from:
                new_det = (id_to,) + det[1:]
                new_detections.append(new_det)
            else:
                new_detections.append(det)
        tracking_data[frame_idx] = new_detections
def count_frames_with_excess(tracking_data, max_expected=4):
    """Cuenta frames con más jugadores del esperado."""
    return sum(1 for dets in tracking_data.values() if len(dets) > max_expected)
def get_player_color(track_id, all_ids, position_x=None, field_center_x=None):
    """
    Genera un color basado en la posición del jugador.
    Azul para jugadores en la mitad izquierda, Rojo para la mitad derecha.
    """
    if position_x is not None and field_center_x is not None:
        if position_x < field_center_x:
            # Mitad izquierda - Azul
            return (255, 0, 0)  # BGR: Azul
        else:
            # Mitad derecha - Rojo
            return (0, 0, 255)  # BGR: Rojo
    
    # Fallback: colores alternos por ID
    colors = [
        (255, 0, 0),      # Azul
        (0, 0, 255),      # Rojo
        (0, 255, 0),      # Verde
        (255, 255, 0),    # Cyan
    ]
    
    sorted_ids = sorted(all_ids)
    if track_id in sorted_ids:
        idx = sorted_ids.index(track_id)
        return colors[idx % len(colors)]
    
    return (255, 255, 255)  # Blanco por defecto
print("✓ Funciones auxiliares definidas")

✓ Funciones auxiliares definidas


## 🛠️ Funciones Auxiliares del Sistema

**Qué hace:** Define todas las funciones auxiliares utilizadas en el pipeline de tracking.

**Funciones principales:**

1. **`get_points()`**: Callback de OpenCV para selección interactiva de puntos con el mouse
2. **`point_in_polygon_with_margin()`**: Verifica si un punto está dentro de un polígono expandido (para zona de detección)
3. **`calculate_iou()`**: Calcula Intersection over Union entre dos bounding boxes (métrica de solapamiento)
4. **`get_track_info()`**: Extrae información completa de una trayectoria (frames, posiciones, bboxes)
5. **`should_merge_ids()`**: Determina si dos IDs deben fusionarse según gaps, distancia e IoU
6. **`merge_track_ids()`**: Fusiona dos IDs de tracking en uno solo
7. **`count_frames_with_excess()`**: Cuenta frames con más jugadores del esperado
8. **`get_player_color()`**: Asigna colores según posición (azul=izquierda, rojo=derecha)

# Bloque 2: Obtención de POI

# Bloque 3: Detección de Jugadores y Corrección de IDs

Este bloque constituye el núcleo del sistema de tracking. Se divide en dos fases principales:

In [4]:
# SIEMPRE seleccionar puntos manualmente (para sobreescribir el JSON)
video = cv2.VideoCapture(VIDEO_PATH)
if not video.isOpened():
    raise RuntimeError(f"No se pudo abrir el video: {VIDEO_PATH}")

fps = video.get(cv2.CAP_PROP_FPS)
total_frames = int(video.get(cv2.CAP_PROP_FRAME_COUNT))
width = int(video.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(video.get(cv2.CAP_PROP_FRAME_HEIGHT))
print(f"✓ Video cargado: {width}x{height}, {fps:.1f} FPS, {total_frames} frames")

ret, first_frame = video.read()
if not ret:
    raise RuntimeError("No se pudo leer el primer frame")

mapa = cv2.imread(MAPA_PATH)
if mapa is None:
    raise FileNotFoundError(f"No se pudo cargar el mapa: {MAPA_PATH}")
print(f"✓ Mapa cargado: {mapa.shape[1]}x{mapa.shape[0]}")

# Seleccionar puntos en el campo (video)
puntos_campo = []
N = 4
imgA = first_frame.copy()
cv2.namedWindow("Selecciona 4 esquinas del campo", cv2.WINDOW_NORMAL)
cv2.resizeWindow("Selecciona 4 esquinas del campo", 1200, 800)
cv2.imshow("Selecciona 4 esquinas del campo", imgA)
cv2.setMouseCallback(
    "Selecciona 4 esquinas del campo",
    get_points,
    {"points": puntos_campo, "image": imgA, 
     "wname": "Selecciona 4 esquinas del campo", "max_points": N}
)
print("Marca las 4 esquinas del campo (arriba-izq, arriba-der, abajo-der, abajo-izq)")
cv2.waitKey(0)
cv2.destroyAllWindows()
puntos_campo = np.array(puntos_campo, dtype=np.float32)
print(f"✓ {len(puntos_campo)} puntos seleccionados en el campo")

# Seleccionar puntos correspondientes en el mapa
puntos_mapa = []

imgB = mapa.copy()
cv2.namedWindow("Selecciona los mismos 4 puntos en el mapa", cv2.WINDOW_NORMAL)
cv2.resizeWindow("Selecciona los mismos 4 puntos en el mapa", 1200, 800)
cv2.imshow("Selecciona los mismos 4 puntos en el mapa", imgB)
cv2.setMouseCallback(
    "Selecciona los mismos 4 puntos en el mapa",
    get_points,
    {"points": puntos_mapa, "image": imgB, 
     "wname": "Selecciona los mismos 4 puntos en el mapa", "max_points": N}
)
print("Marca los mismos 4 puntos en el MAPA (mismo orden)")
cv2.waitKey(0)
cv2.destroyAllWindows()
puntos_mapa = np.array(puntos_mapa, dtype=np.float32)
print(f"✓ {len(puntos_mapa)} puntos seleccionados en el mapa")

# Seleccionar zona de detección (arena completa)
print("\nAhora marca las 4 esquinas de la ZONA DE DETECCIÓN (toda la arena)")
print("Esta área debe ser más grande que el campo para no perder jugadores en los bordes\n")

video.set(cv2.CAP_PROP_POS_FRAMES, 0)
ret, frame_arena = video.read()
if not ret:
    raise RuntimeError("No se pudo leer el frame para seleccionar zona de detección")

puntos_arena = []
imgC = frame_arena.copy()

# Dibujar el campo ya seleccionado como referencia
for i, pt in enumerate(puntos_campo, 1):
    cv2.circle(imgC, tuple(pt.astype(int)), 8, (0, 255, 0), -1)
    cv2.putText(imgC, f"Campo {i}", (int(pt[0]) + 10, int(pt[1]) - 10),
               cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 2)
cv2.polylines(imgC, [puntos_campo.astype(int)], True, (0, 255, 0), 2)

cv2.namedWindow("Selecciona 4 esquinas de la ZONA DE DETECCION (arena)", cv2.WINDOW_NORMAL)
cv2.resizeWindow("Selecciona 4 esquinas de la ZONA DE DETECCION (arena)", 1200, 800)
cv2.imshow("Selecciona 4 esquinas de la ZONA DE DETECCION (arena)", imgC)
cv2.setMouseCallback(
    "Selecciona 4 esquinas de la ZONA DE DETECCION (arena)",
    get_points,
    {"points": puntos_arena, "image": imgC, 
     "wname": "Selecciona 4 esquinas de la ZONA DE DETECCION (arena)", "max_points": N}
)
print("Marca las 4 esquinas de la arena (más grandes que el campo verde)")
cv2.waitKey(0)
cv2.destroyAllWindows()
puntos_arena = np.array(puntos_arena, dtype=np.float32)
print(f"✓ {len(puntos_arena)} puntos seleccionados en la zona de detección")

video.release()

# Guardar puntos en JSON (sobreescribir)
puntos_data = {
    'puntos_campo': puntos_campo.tolist(),
    'puntos_mapa': puntos_mapa.tolist(),
    'puntos_arena': puntos_arena.tolist()
}

with open(PUNTOS_MAPA_JSON, 'w') as f:
    json.dump(puntos_data, f, indent=2)

print(f"✓ Puntos y zona de detección guardados en '{PUNTOS_MAPA_JSON}'")

# Calcular matriz de homografía
H, status = cv2.findHomography(puntos_campo, puntos_mapa)
print(f"✓ Matriz de homografía calculada")

# Calcular centro del campo en video para determinar colores
field_center_x = int(np.mean(puntos_campo[:, 0]))
field_center_y = int(np.mean(puntos_campo[:, 1]))
print(f"✓ Centro del campo: ({field_center_x}, {field_center_y})")

✓ Video cargado: 1920x1080, 30.0 FPS, 490 frames
✓ Mapa cargado: 1536x1024
Marca las 4 esquinas del campo (arriba-izq, arriba-der, abajo-der, abajo-izq)
✓ 4 puntos seleccionados en el campo
Marca los mismos 4 puntos en el MAPA (mismo orden)
✓ 4 puntos seleccionados en el mapa

Ahora marca las 4 esquinas de la ZONA DE DETECCIÓN (toda la arena)
Esta área debe ser más grande que el campo para no perder jugadores en los bordes

Marca las 4 esquinas de la arena (más grandes que el campo verde)
✓ 4 puntos seleccionados en la zona de detección
✓ Puntos y zona de detección guardados en 'Output/puntos_mapa.json'
✓ Matriz de homografía calculada
✓ Centro del campo: (986, 818)


## 📍 Selección de Puntos de Referencia (POI)

**Qué hace:** Permite seleccionar interactivamente los puntos de referencia necesarios para la homografía y la zona de detección.

**Flujo del proceso:**

1. **Campo (4 puntos)**: Selecciona las 4 esquinas del campo en el video
   - Orden: arriba-izq → arriba-der → abajo-der → abajo-izq
   - Usado para calcular la homografía

2. **Mapa (4 puntos)**: Selecciona los mismos 4 puntos en la imagen del campo
   - Mismo orden que en el video
   - Permite transformar coordenadas video → mapa

3. **Arena (4 puntos)**: Define la zona de detección completa
   - Debe ser más grande que el campo
   - Evita perder jugadores cerca de los bordes
   - Muestra el campo en verde como referencia

**Salida:**
- Guarda `puntos_campo`, `puntos_mapa`, `puntos_arena` en JSON
- Calcula matriz de homografía **H**
- Determina el centro del campo para asignación de colores por equipo

In [11]:
# ============================================================================
# Bloque 3: Detección de Jugadores y Corrección de IDs
# ============================================================================

print("\n" + "=" * 70)
print("FASE 1: DETECCIÓN DE JUGADORES")
print("=" * 70)

video = cv2.VideoCapture(VIDEO_PATH)
video.set(cv2.CAP_PROP_POS_FRAMES, 0)
tracking_data = {}

print(f"\nProcesando {total_frames} frames...")
print("Esto puede tardar unos minutos...\n")

frame_idx = 0

while True:
    ret, frame = video.read()
    if not ret:
        break
    
    results = model.track(frame, persist=True, verbose=False, classes=[0])
    frame_detections = []
    
    for r in results:
        if r.boxes.id is None:
            continue
            
        for box, track_id in zip(r.boxes, r.boxes.id):
            x1, y1, x2, y2 = map(int, box.xyxy[0])
            cx = (x1 + x2) // 2
            cy = y2
            
            # Verificar que esté dentro de la zona de detección (arena)
            in_arena = cv2.pointPolygonTest(puntos_arena.astype(np.int32), (cx, cy), False) >= 0
            
            if in_arena:
                frame_detections.append((
                    int(track_id.item()),
                    x1, y1, x2, y2,
                    cx, cy
                ))
    
    tracking_data[frame_idx] = frame_detections
    
    if frame_idx % 50 == 0:
        progress = (frame_idx / total_frames) * 100
        print(f"  Frame {frame_idx}/{total_frames} ({progress:.1f}%) - {len(frame_detections)} jugadores detectados")
    
    frame_idx += 1

video.release()

print(f"\n✓ Tracking completado: {len(tracking_data)} frames procesados")

# Estadísticas iniciales
all_track_ids = set()
for dets in tracking_data.values():
    for det in dets:
        all_track_ids.add(det[0])

total_detections = sum(len(dets) for dets in tracking_data.values())
print(f"  Total detecciones: {total_detections}")
print(f"  IDs únicos detectados: {len(all_track_ids)}")
print(f"  IDs: {sorted(all_track_ids)}")

# ============================================================================
# FASE 2: CORRECCIÓN INTELIGENTE DE IDS (Fusión + Reasignación + Interpolación)
# ============================================================================

print("\n" + "=" * 70)
print("FASE 2: CORRECCIÓN INTELIGENTE DE IDS")
print("=" * 70)

all_ids = sorted(all_track_ids)
print(f"\n📊 Estado inicial:")
print(f"   IDs detectados: {all_ids}")
print(f"   Total IDs: {len(all_ids)}")
print(f"   Frames con >4 jugadores: {count_frames_with_excess(tracking_data)}")

# SUBFASE 1: Fusionar IDs fragmentados del mismo jugador
print(f"\n{'=' * 70}")
print("SUBFASE 1: FUSIÓN DE IDS FRAGMENTADOS")
print(f"{'=' * 70}")

merge_map = {id: id for id in all_ids}
total_merges = 0

params = [
    (5, 100, 0.4, "Muy estricto - gaps pequeños"),
    (10, 150, 0.3, "Estricto - gaps medianos"),
    (15, 200, 0.25, "Moderado - gaps más largos"),
    (20, 250, 0.2, "Permisivo - oclusiones largas"),
]

for iteration, (max_gap, max_distance, min_iou, description) in enumerate(params, 1):
    print(f"\n{'─' * 70}")
    print(f"Iteración {iteration}: {description}")
    print(f"   Parámetros: gap≤{max_gap}f, dist≤{max_distance}px, IoU≥{min_iou}")
    
    current_ids = set()
    for dets in tracking_data.values():
        for det in dets:
            current_ids.add(det[0])
    current_ids = sorted(current_ids)
    
    frames_excess = count_frames_with_excess(tracking_data)
    print(f"   IDs actuales: {len(current_ids)} | Frames con >4: {frames_excess}")
    
    merge_candidates = []
    
    for i, id1 in enumerate(current_ids):
        for id2 in current_ids[i+1:]:
            if should_merge_ids(tracking_data, id1, id2, max_gap, max_distance, min_iou):
                impact = 0
                for frame_idx, dets in tracking_data.items():
                    ids_in_frame = [d[0] for d in dets]
                    if id1 in ids_in_frame and id2 in ids_in_frame:
                        impact += 1
                
                merge_candidates.append((id1, id2, impact))
    
    merge_candidates.sort(key=lambda x: x[2], reverse=True)
    
    if not merge_candidates:
        print(f"   No se encontraron fusiones")
        continue
    
    merges_in_iteration = 0
    for id1, id2, impact in merge_candidates:
        current_id1 = merge_map.get(id1, id1)
        current_id2 = merge_map.get(id2, id2)
        
        if current_id1 == current_id2:
            continue
        
        merge_from = max(current_id1, current_id2)
        merge_to = min(current_id1, current_id2)
        
        print(f"      → Fusionando ID {merge_from} → ID {merge_to}")
        
        merge_track_ids(tracking_data, merge_from, merge_to)
        
        for key in merge_map:
            if merge_map[key] == merge_from:
                merge_map[key] = merge_to
        merge_map[merge_from] = merge_to
        
        merges_in_iteration += 1
        total_merges += 1
    
    print(f"   ✓ Fusiones: {merges_in_iteration}")

# SUBFASE 2: Reasignar IDs duplicados y eliminar detecciones extras
print(f"\n{'=' * 70}")
print("SUBFASE 2: REASIGNACIÓN Y LIMPIEZA DE IDS")
print(f"{'=' * 70}")

# Identificar los 4 IDs principales (más presentes)
id_frame_counts = defaultdict(int)
for dets in tracking_data.values():
    for det in dets:
        id_frame_counts[det[0]] += 1

# Los 4 IDs con más frames son los principales
main_ids = sorted(id_frame_counts.items(), key=lambda x: x[1], reverse=True)[:4]
main_ids = [id for id, count in main_ids]

print(f"\n🎯 IDs principales identificados: {main_ids}")
print(f"   Apariciones:")
for id in main_ids:
    count = id_frame_counts[id]
    percentage = (count / len(tracking_data)) * 100
    print(f"      ID {id}: {count} frames ({percentage:.1f}%)")

# Construir trayectorias de los IDs principales
main_trajectories = {id: [] for id in main_ids}
for frame_idx in sorted(tracking_data.keys()):
    for det in tracking_data[frame_idx]:
        track_id = det[0]
        if track_id in main_ids:
            cx, cy = det[5], det[6]
            main_trajectories[track_id].append((frame_idx, cx, cy))

print(f"\n🔧 Procesando todos los frames...")

reassignments = 0
duplicates_removed = 0
extras_removed = 0

for frame_idx in sorted(tracking_data.keys()):
    detections = tracking_data[frame_idx]
    
    # PASO 1: Eliminar IDs que NO son principales
    filtered_detections = []
    for det in detections:
        if det[0] in main_ids:
            filtered_detections.append(det)
        else:
            extras_removed += 1
            print(f"   Frame {frame_idx}: ID {det[0]} eliminado (no es principal)")
    
    # PASO 2: Resolver duplicados del mismo ID
    id_groups = defaultdict(list)
    for det in filtered_detections:
        id_groups[det[0]].append(det)
    
    clean_detections = []
    for track_id, dets_list in id_groups.items():
        if len(dets_list) == 1:
            clean_detections.append(dets_list[0])
        else:
            # Hay duplicados del mismo ID - mantener el más coherente con la trayectoria
            trajectory = main_trajectories.get(track_id, [])
            
            # Buscar última posición conocida
            last_pos = None
            for traj_frame, traj_cx, traj_cy in reversed(trajectory):
                if traj_frame < frame_idx:
                    last_pos = (traj_cx, traj_cy)
                    break
            
            if last_pos:
                # Elegir la detección más cercana a la trayectoria
                best_det = min(dets_list, 
                             key=lambda d: np.sqrt((d[5]-last_pos[0])**2 + (d[6]-last_pos[1])**2))
                clean_detections.append(best_det)
                duplicates_removed += len(dets_list) - 1
                print(f"   Frame {frame_idx}: {len(dets_list)-1} duplicados del ID {track_id} eliminados")
            else:
                # Sin trayectoria previa, mantener el primero
                clean_detections.append(dets_list[0])
                duplicates_removed += len(dets_list) - 1
    
    # PASO 3: Si faltan IDs, intentar reasignar desde IDs no principales en el frame original
    if len(clean_detections) < EXPECTED_PLAYERS:
        present_main_ids = {det[0] for det in clean_detections}
        missing_main_ids = [id for id in main_ids if id not in present_main_ids]
        
        # Buscar detecciones no principales en el frame original
        extra_dets = [det for det in detections if det[0] not in main_ids]
        
        for extra_det in extra_dets:
            if not missing_main_ids:
                break
            
            extra_cx, extra_cy = extra_det[5], extra_det[6]
            
            # Encontrar el mejor ID faltante para esta detección
            best_id = None
            min_distance = float('inf')
            MAX_REASSIGN_DISTANCE = 200  # Máxima distancia para reasignación (evita saltos)
            
            for missing_id in missing_main_ids:
                trajectory = main_trajectories[missing_id]
                if not trajectory:
                    continue
                
                # Buscar última posición antes de este frame
                last_pos = None
                for traj_frame, traj_cx, traj_cy in reversed(trajectory):
                    if traj_frame < frame_idx:
                        last_pos = (traj_cx, traj_cy)
                        break
                
                if last_pos:
                    distance = np.sqrt((extra_cx - last_pos[0])**2 + 
                                     (extra_cy - last_pos[1])**2)
                    if distance < min_distance and distance < MAX_REASSIGN_DISTANCE:
                        min_distance = distance
                        best_id = missing_id
            
            if best_id is not None and min_distance < MAX_REASSIGN_DISTANCE:
                # Reasignar detección al mejor ID
                new_det = (best_id,) + extra_det[1:]
                clean_detections.append(new_det)
                
                # Actualizar trayectoria
                main_trajectories[best_id].append((frame_idx, extra_cx, extra_cy))
                
                # Remover de la lista de faltantes
                missing_main_ids.remove(best_id)
                
                reassignments += 1
                print(f"   Frame {frame_idx}: ID {extra_det[0]} → ID {best_id} (dist: {min_distance:.0f}px)")
            else:
                if best_id is not None:
                    print(f"   Frame {frame_idx}: ID {extra_det[0]} descartado (distancia {min_distance:.0f}px > {MAX_REASSIGN_DISTANCE}px)")
    
    # Actualizar frame con detecciones limpias
    tracking_data[frame_idx] = clean_detections

print(f"\n✓ Estadísticas de limpieza:")
print(f"   IDs no principales eliminados: {extras_removed}")
print(f"   Duplicados eliminados: {duplicates_removed}")
print(f"   Reasignaciones: {reassignments}")

# SUBFASE 3: Completar frames faltantes con interpolación
print(f"\n{'=' * 70}")
print("SUBFASE 3: INTERPOLACIÓN DE FRAMES FALTANTES")
print(f"{'=' * 70}")

interpolations = 0

for main_id in main_ids:
    trajectory = main_trajectories[main_id]
    if len(trajectory) < 2:
        continue
    
    # Ordenar trayectoria
    trajectory = sorted(trajectory, key=lambda x: x[0])
    
    # Buscar gaps pequeños (<10 frames)
    for i in range(len(trajectory) - 1):
        frame1, cx1, cy1 = trajectory[i]
        frame2, cx2, cy2 = trajectory[i + 1]
        
        gap = frame2 - frame1
        if gap > 1 and gap <= 10:
            # Interpolación lineal
            for intermediate_frame in range(frame1 + 1, frame2):
                # Verificar que no exista ya esta detección
                existing_ids = [det[0] for det in tracking_data.get(intermediate_frame, [])]
                if main_id not in existing_ids:
                    # Interpolar posición
                    t = (intermediate_frame - frame1) / gap
                    interp_cx = int(cx1 + t * (cx2 - cx1))
                    interp_cy = int(cy1 + t * (cy2 - cy1))
                    
                    # Verificar que esté dentro del campo
                    if point_in_polygon_with_margin((interp_cx, interp_cy), puntos_campo, MARGIN_PERCENT):
                        # Estimar bbox
                        bbox_size = 50  # tamaño aproximado
                        x1 = interp_cx - bbox_size // 2
                        y1 = interp_cy - bbox_size
                        x2 = interp_cx + bbox_size // 2
                        y2 = interp_cy
                        
                        interpolated_det = (main_id, x1, y1, x2, y2, interp_cx, interp_cy)
                        
                        if intermediate_frame not in tracking_data:
                            tracking_data[intermediate_frame] = []
                        tracking_data[intermediate_frame].append(interpolated_det)
                        interpolations += 1

print(f"✓ Interpolaciones: {interpolations} detecciones añadidas")

# Estadísticas finales
print(f"\n{'=' * 70}")
print("✅ CORRECCIÓN COMPLETADA")
print(f"{'=' * 70}")

final_ids = set()
for dets in tracking_data.values():
    for det in dets:
        final_ids.add(det[0])
final_ids = sorted(final_ids)

print(f"\n📊 Resumen de cambios:")
print(f"   IDs originales: {len(all_ids)} → IDs finales: {len(final_ids)}")
print(f"   Fusiones realizadas: {total_merges}")
print(f"   Reasignaciones: {reassignments}")
print(f"   Interpolaciones: {interpolations}")
print(f"   IDs finales: {final_ids}")

print(f"\n📈 Distribución de jugadores por frame:")
distribution = defaultdict(int)
for dets in tracking_data.values():
    distribution[len(dets)] += 1

for num_players in sorted(distribution.keys()):
    count = distribution[num_players]
    percentage = (count / len(tracking_data)) * 100
    bar = "█" * int(percentage / 2)
    marker = "✓" if num_players == EXPECTED_PLAYERS else "⚠" if num_players > EXPECTED_PLAYERS else "!"
    print(f"   {marker} {num_players} jugadores: {count:4d} frames ({percentage:5.1f}%) {bar}")

frames_exact = distribution.get(EXPECTED_PLAYERS, 0)
frames_over = sum(count for num, count in distribution.items() if num > EXPECTED_PLAYERS)
frames_under = sum(count for num, count in distribution.items() if num < EXPECTED_PLAYERS)

print(f"\n🎯 Calidad del tracking:")
print(f"   Frames perfectos (4 jugadores): {frames_exact} ({frames_exact/len(tracking_data)*100:.1f}%)")
print(f"   Frames con exceso (>4): {frames_over} ({frames_over/len(tracking_data)*100:.1f}%)")
print(f"   Frames con déficit (<4): {frames_under} ({frames_under/len(tracking_data)*100:.1f}%)")

if len(final_ids) == EXPECTED_PLAYERS:
    print(f"\n🎉 ¡PERFECTO! Exactamente {EXPECTED_PLAYERS} jugadores detectados")
elif len(final_ids) < EXPECTED_PLAYERS:
    print(f"\n⚠️  Solo {len(final_ids)} jugadores detectados (esperados: {EXPECTED_PLAYERS})")
else:
    print(f"\n⚠️  {len(final_ids)} jugadores detectados (esperados: {EXPECTED_PLAYERS})")

all_track_ids = final_ids


FASE 1: DETECCIÓN DE JUGADORES

Procesando 490 frames...
Esto puede tardar unos minutos...

  Frame 0/490 (0.0%) - 5 jugadores detectados
  Frame 50/490 (10.2%) - 4 jugadores detectados
  Frame 100/490 (20.4%) - 4 jugadores detectados
  Frame 150/490 (30.6%) - 4 jugadores detectados
  Frame 200/490 (40.8%) - 5 jugadores detectados
  Frame 250/490 (51.0%) - 3 jugadores detectados
  Frame 300/490 (61.2%) - 6 jugadores detectados
  Frame 350/490 (71.4%) - 5 jugadores detectados
  Frame 400/490 (81.6%) - 3 jugadores detectados

✓ Tracking completado: 441 frames procesados
  Total detecciones: 1757
  IDs únicos detectados: 27
  IDs: [2, 3, 4, 5, 6, 16, 17, 26, 27, 29, 33, 48, 55, 60, 65, 69, 70, 71, 87, 88, 104, 114, 143, 159, 161, 174, 188]

FASE 2: CORRECCIÓN INTELIGENTE DE IDS

📊 Estado inicial:
   IDs detectados: [2, 3, 4, 5, 6, 16, 17, 26, 27, 29, 33, 48, 55, 60, 65, 69, 70, 71, 87, 88, 104, 114, 143, 159, 161, 174, 188]
   Total IDs: 27
   Frames con >4 jugadores: 100

SUBFASE 1: FUS

## 🎯 Detección y Corrección Inteligente de IDs

**Qué hace:** Detecta jugadores en cada frame y corrige los errores de tracking mediante un sistema de 3 fases.

---

### **FASE 1: DETECCIÓN INICIAL**
- Procesa todos los frames del video con YOLO11n
- Filtra detecciones: solo personas (clase 0) dentro de la arena
- Extrae: ID, bounding box, centro (cx, cy)
- Almacena en `tracking_data[frame_idx]`

---

### **FASE 2: CORRECCIÓN DE IDS**

#### **SUBFASE 2.1: Fusión de IDs Fragmentados**
**Problema:** Un mismo jugador puede recibir múltiples IDs por oclusiones  
**Solución:** Fusiona IDs que probablemente pertenecen al mismo jugador

**Criterios de fusión:**
- **Secuencial**: IDs que aparecen en momentos distintos pero cercanos
  - Gap temporal pequeño (≤5-20 frames)
  - Distancia espacial razonable (≤100-250px)
  - Tamaño de bbox similar
- **Solapamiento**: IDs que coexisten en tiempo
  - IoU alto (≥0.3)
  - Distancia muy pequeña

**Proceso:**
1. 4 iteraciones con parámetros progresivamente más permisivos
2. Prioriza fusiones que reducen duplicados en frames
3. Mantiene mapa de fusiones para consistencia

#### **SUBFASE 2.2: Reasignación y Limpieza**
**Problema:** Múltiples detecciones del mismo ID en un frame, o IDs extras  
**Solución:** Identifica los 4 IDs principales y limpia el resto

**Proceso:**
1. **Identificar IDs principales**: Los 4 con más apariciones
2. **Eliminar IDs no principales**: Detecciones esporádicas
3. **Resolver duplicados**: Si un ID aparece 2+ veces en un frame, mantener el más coherente con la trayectoria
4. **Reasignar IDs faltantes**: Si faltan IDs principales, buscar en detecciones descartadas y reasignar si están cerca (<200px)

#### **SUBFASE 2.3: Interpolación**
**Problema:** Frames donde falta un jugador (< 4 detecciones)  
**Solución:** Interpola posiciones en gaps pequeños (≤10 frames)

**Proceso:**
- Interpolación lineal de posición (cx, cy)
- Verifica que esté dentro del campo
- Estima bbox de tamaño fijo (50x50px)

---

### **Salida:**
- `tracking_data`: Diccionario frame → lista de detecciones corregidas
- `final_ids`: Lista de los 4 IDs finales
- Estadísticas de calidad (frames perfectos, excesos, déficits)

# Bloque 4.1: Exportación de Resultados

In [12]:
print(f"\n{'=' * 70}")
print("EXPORTANDO DATOS")
print(f"{'=' * 70}\n")

tracking_list = []

for frame_idx, detections in sorted(tracking_data.items()):
    for det in detections:
        track_id, x1, y1, x2, y2, cx, cy = det
        
        # Transformar coordenadas del video al mapa usando homografía
        point_video = np.array([[[cx, cy]]], dtype=np.float32)
        point_mapa = cv2.perspectiveTransform(point_video, H)
        field_x, field_y = point_mapa[0][0]
        
        tracking_list.append({
            'frame': frame_idx,
            'track_id': track_id,
            'bbox_x1': x1,
            'bbox_y1': y1,
            'bbox_x2': x2,
            'bbox_y2': y2,
            'center_x': cx,
            'center_y': cy,
            'field_x': float(field_x),
            'field_y': float(field_y),
            'timestamp_sec': frame_idx / fps
        })

csv_filename = "Output/tracking_data.csv"
with open(csv_filename, 'w', newline='', encoding='utf-8') as csvfile:
    fieldnames = ['frame', 'track_id', 'bbox_x1', 'bbox_y1', 'bbox_x2', 'bbox_y2', 
                  'center_x', 'center_y', 'field_x', 'field_y', 'timestamp_sec']
    writer = csv.DictWriter(csvfile, fieldnames=fieldnames)
    writer.writeheader()
    writer.writerows(tracking_list)

print(f"✓ Datos exportados a '{csv_filename}'")
print(f"  Total de registros: {len(tracking_list)}")

summary_data = {
    'video_info': {
        'fps': fps,
        'total_frames': total_frames,
        'width': width,
        'height': height
    },
    'campo_puntos': puntos_campo.tolist(),
    'mapa_puntos': puntos_mapa.tolist(),
    'arena_puntos': puntos_arena.tolist(),
    'homography_matrix': H.tolist(),
    'jugadores_ids': sorted(final_ids),
    'num_jugadores': len(final_ids),
    'total_detecciones': len(tracking_list),
    'correccion_stats': {
        'fusiones': total_merges,
        'extras_eliminados': extras_removed,
        'duplicados_eliminados': duplicates_removed,
        'reasignaciones': reassignments,
        'interpolaciones': interpolations
    }
}

json_filename = "Output/tracking_summary.json"
with open(json_filename, 'w', encoding='utf-8') as jsonfile:
    json.dump(summary_data, jsonfile, indent=2)

print(f"✓ Resumen exportado a '{json_filename}'")

print(f"\n📁 Archivos generados:")
print(f"   • {csv_filename}")
print(f"   • {json_filename}")



EXPORTANDO DATOS

✓ Datos exportados a 'Output/tracking_data.csv'
  Total de registros: 1131
✓ Resumen exportado a 'Output/tracking_summary.json'

📁 Archivos generados:
   • Output/tracking_data.csv
   • Output/tracking_summary.json


## 💾 Exportación de Datos de Tracking

**Qué hace:** Exporta los datos de tracking procesados a formatos estándar (CSV y JSON).

**Proceso:**

1. **Transformación de coordenadas:**
   - Para cada detección, transforma (cx, cy) del video al mapa usando la homografía **H**
   - Calcula timestamp en segundos: `frame / fps`

2. **Exportación CSV (`tracking_data.csv`):**
   - Formato tabular con todas las detecciones
   - Columnas: frame, track_id, bbox (x1,y1,x2,y2), center (x,y), field (x,y), timestamp
   - Útil para análisis en pandas, Excel, etc.

3. **Exportación JSON (`tracking_summary.json`):**
   - Metadatos completos del tracking
   - Info del video: fps, resolución, frames totales
   - Puntos de referencia: campo, mapa, arena
   - Matriz de homografía
   - IDs de jugadores detectados
   - Estadísticas de corrección: fusiones, reasignaciones, interpolaciones

**Salida:**
- `Output/tracking_data.csv`: Datos frame a frame
- `Output/tracking_summary.json`: Resumen y metadatos

# Bloque 4.2: Visualización de Resultados

## Generación de Video Combinado

In [13]:
print(f"\n{'=' * 70}")
print("GENERANDO VIDEO COMBINADO")
print(f"{'=' * 70}\n")

# Cargar video y mapa
video = cv2.VideoCapture(VIDEO_PATH)
mapa = cv2.imread(MAPA_PATH)

# Configuración del video de salida
output_filename = "Output/player_combined_tracking.mp4"
fourcc = cv2.VideoWriter_fourcc(*'mp4v')

# Dimensiones: video original arriba, mapa abajo (centrado y más pequeño)
map_display_width = width // 2  # Mapa será la mitad del ancho del video
map_display_height = int(mapa.shape[0] * (map_display_width / mapa.shape[1]))

# Altura total: video + mapa
output_height = height + map_display_height
output_width = width

out = cv2.VideoWriter(output_filename, fourcc, fps, (output_width, output_height))

# Obtener todos los IDs para colores consistentes
all_ids = sorted(final_ids)

print(f"Generando video combinado...")
print(f"  Resolución: {output_width}x{output_height}")
print(f"  FPS: {fps}")

frame_idx = 0
video.set(cv2.CAP_PROP_POS_FRAMES, 0)

# Diccionario para almacenar trayectorias
trajectories = defaultdict(list)

while True:
    ret, frame = video.read()
    if not ret:
        break
    
    # ===== PARTE SUPERIOR: VIDEO CON TRACKING =====
    tracking_frame = frame.copy()
    
    # Dibujar zona de detección (arena) en amarillo y campo en verde
    cv2.polylines(tracking_frame, [puntos_arena.astype(np.int32)], True, (0, 255, 255), 3)  # Arena en amarillo
    cv2.polylines(tracking_frame, [puntos_campo.astype(np.int32)], True, (0, 255, 0), 2)    # Campo en verde
    
    # Dibujar detecciones
    if frame_idx in tracking_data:
        detections = tracking_data[frame_idx]
        
        for det in detections:
            track_id, x1, y1, x2, y2, cx, cy = det
            
            # Obtener color basado en posición (izquierda/derecha)
            color = get_player_color(track_id, all_ids, cx, field_center_x)
            
            # Dibujar bounding box y centro
            cv2.rectangle(tracking_frame, (x1, y1), (x2, y2), color, 2)
            cv2.circle(tracking_frame, (cx, cy), 6, color, -1)
            cv2.putText(tracking_frame, f"ID {track_id}", (x1, y1 - 10),
                       cv2.FONT_HERSHEY_SIMPLEX, 0.7, color, 2)
            
            # Guardar posición en mapa para trayectoria (para el PNG final)
            point_video = np.array([[[cx, cy]]], dtype=np.float32)
            point_mapa = cv2.perspectiveTransform(point_video, H)
            field_x, field_y = point_mapa[0][0]
            trajectories[track_id].append((int(field_x), int(field_y), cx))  # Añadir cx para color
    
    # Info del frame
    info_text = f"Frame: {frame_idx}/{total_frames} | Jugadores: {len(tracking_data.get(frame_idx, []))}"
    cv2.putText(tracking_frame, info_text, (10, 30),
               cv2.FONT_HERSHEY_SIMPLEX, 0.8, (255, 255, 255), 2)
    
    # ===== PARTE INFERIOR: VISTA DE HOMOGRAFÍA =====
    # Redimensionar mapa
    mapa_resized = cv2.resize(mapa, (map_display_width, map_display_height))
    
    # Dibujar solo posiciones actuales en el mapa (SIN trayectorias)
    mapa_display = mapa_resized.copy()
    
    # Obtener posiciones actuales del frame
    if frame_idx in tracking_data:
        detections = tracking_data[frame_idx]
        
        for det in detections:
            track_id, x1, y1, x2, y2, cx, cy = det
            
            # Obtener color basado en posición
            color = get_player_color(track_id, all_ids, cx, field_center_x)
            
            # Transformar posición al mapa
            point_video = np.array([[[cx, cy]]], dtype=np.float32)
            point_mapa_pos = cv2.perspectiveTransform(point_video, H)
            field_x, field_y = point_mapa_pos[0][0]
            
            # Escalar al mapa redimensionado
            scale_x = map_display_width / mapa.shape[1]
            scale_y = map_display_height / mapa.shape[0]
            
            current_pos = (int(field_x * scale_x), int(field_y * scale_y))
            
            # Dibujar posición actual
            cv2.circle(mapa_display, current_pos, 8, color, -1)
            cv2.circle(mapa_display, current_pos, 10, (255, 255, 255), 2)
            cv2.putText(mapa_display, f"{track_id}", 
                       (current_pos[0] + 12, current_pos[1] - 12),
                       cv2.FONT_HERSHEY_SIMPLEX, 0.6, color, 2)
    
    # ===== COMBINAR AMBAS PARTES =====
    # Crear frame de salida
    combined_frame = np.zeros((output_height, output_width, 3), dtype=np.uint8)
    
    # Colocar video tracking arriba
    combined_frame[0:height, 0:width] = tracking_frame
    
    # Colocar mapa abajo centrado
    map_x_offset = (output_width - map_display_width) // 2
    combined_frame[height:height+map_display_height, 
                   map_x_offset:map_x_offset+map_display_width] = mapa_display
    
    # Escribir frame
    out.write(combined_frame)
    
    if frame_idx % 50 == 0:
        progress = (frame_idx / total_frames) * 100
        print(f"  Procesado: {frame_idx}/{total_frames} ({progress:.1f}%)")
    
    frame_idx += 1

video.release()
out.release()

print(f"\n✓ Video combinado generado: '{output_filename}'")
print(f"  Frames procesados: {frame_idx}")
print(f"  Duración: {frame_idx/fps:.1f} segundos")



GENERANDO VIDEO COMBINADO

Generando video combinado...
  Resolución: 1920x1720
  FPS: 30.0
  Procesado: 0/490 (0.0%)
  Procesado: 50/490 (10.2%)
  Procesado: 100/490 (20.4%)
  Procesado: 150/490 (30.6%)
  Procesado: 200/490 (40.8%)
  Procesado: 250/490 (51.0%)
  Procesado: 300/490 (61.2%)
  Procesado: 350/490 (71.4%)
  Procesado: 400/490 (81.6%)

✓ Video combinado generado: 'Output/player_combined_tracking.mp4'
  Frames procesados: 441
  Duración: 14.7 segundos


## 🎥 Generación de Video Combinado con Tracking

**Qué hace:** Crea un video que combina la vista del tracking en el video original (arriba) con la vista transformada en el mapa (abajo).

**Proceso:**

### **Parte Superior: Video con Tracking**
1. Dibuja la zona de detección (arena) en **amarillo** (línea gruesa)
2. Dibuja el campo en **verde** (línea fina) como referencia
3. Para cada jugador detectado:
   - Dibuja bounding box en color del equipo (azul=izquierda, rojo=derecha)
   - Marca el centro con un círculo
   - Muestra el ID del jugador
4. Overlay con info del frame: número y cantidad de jugadores

### **Parte Inferior: Vista de Mapa (Homografía)**
1. Redimensiona el mapa a la mitad del ancho del video
2. Transforma posiciones de jugadores usando la homografía **H**
3. Escala al tamaño del mapa redimensionado
4. Dibuja cada jugador como:
   - Círculo relleno (color del equipo)
   - Círculo blanco exterior
   - ID del jugador

### **Combinación:**
- Video arriba (altura original)
- Mapa abajo centrado (mitad del ancho)
- Codec: MP4V
- Mismo framerate que el video original

**Salida:**
- `Output/player_combined_tracking.mp4`: Video con tracking y vista de mapa

## Generación de Imagen de Trayectorias

In [14]:
print(f"\n{'=' * 70}")
print("GENERANDO IMAGEN DE TRAYECTORIAS")
print(f"{'=' * 70}\n")

# Cargar mapa original
mapa_traj = cv2.imread(MAPA_PATH)

print("Dibujando trayectorias completas...")

# Reconstruir trayectorias completas con información de posición para colores
full_trajectories = defaultdict(list)

for frame_idx in sorted(tracking_data.keys()):
    if frame_idx not in tracking_data:
        continue
    
    for det in tracking_data[frame_idx]:
        track_id, x1, y1, x2, y2, cx, cy = det
        
        # Transformar al mapa
        point_video = np.array([[[cx, cy]]], dtype=np.float32)
        point_mapa = cv2.perspectiveTransform(point_video, H)
        field_x, field_y = point_mapa[0][0]
        
        full_trajectories[track_id].append((int(field_x), int(field_y), cx))

# Dibujar cada trayectoria
for track_id in sorted(full_trajectories.keys()):
    trajectory = full_trajectories[track_id]
    
    if len(trajectory) < 2:
        continue
    
    print(f"  Dibujando trayectoria del ID {track_id}: {len(trajectory)} puntos")
    
    # Dibujar líneas de trayectoria
    for i in range(1, len(trajectory)):
        field_x1, field_y1, cx1 = trajectory[i-1]
        field_x2, field_y2, cx2 = trajectory[i]
        
        # Determinar color basado en la posición promedio del segmento
        avg_cx = (cx1 + cx2) / 2
        color = get_player_color(track_id, all_ids, avg_cx, field_center_x)
        
        # Dibujar línea
        cv2.line(mapa_traj, (field_x1, field_y1), (field_x2, field_y2), color, 3)
    
    # Marcar inicio y fin
    start_x, start_y, start_cx = trajectory[0]
    end_x, end_y, end_cx = trajectory[-1]
    
    start_color = get_player_color(track_id, all_ids, start_cx, field_center_x)
    end_color = get_player_color(track_id, all_ids, end_cx, field_center_x)
    
    # Punto de inicio (círculo blanco con borde)
    cv2.circle(mapa_traj, (start_x, start_y), 12, (255, 255, 255), -1)
    cv2.circle(mapa_traj, (start_x, start_y), 8, start_color, -1)
    cv2.putText(mapa_traj, f"{track_id}", (start_x - 10, start_y - 15),
               cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 255), 2)
    
    # Punto final (cuadrado)
    cv2.rectangle(mapa_traj, (end_x-8, end_y-8), (end_x+8, end_y+8), end_color, -1)
    cv2.rectangle(mapa_traj, (end_x-10, end_y-10), (end_x+10, end_y+10), (255, 255, 255), 2)

# Añadir leyenda
legend_y = 30
cv2.putText(mapa_traj, "EQUIPO IZQUIERDO", (20, legend_y),
           cv2.FONT_HERSHEY_SIMPLEX, 0.8, (255, 0, 0), 2)  # Azul
cv2.putText(mapa_traj, "EQUIPO DERECHO", (20, legend_y + 35),
           cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 0, 255), 2)  # Rojo

# Guardar imagen
traj_filename = "Output/trayectorias.png"
cv2.imwrite(traj_filename, mapa_traj)

print(f"\n✓ Imagen de trayectorias generada: '{traj_filename}'")
print(f"  Total de trayectorias dibujadas: {len(full_trajectories)}")



GENERANDO IMAGEN DE TRAYECTORIAS

Dibujando trayectorias completas...
  Dibujando trayectoria del ID 2: 249 puntos
  Dibujando trayectoria del ID 3: 441 puntos
  Dibujando trayectoria del ID 6: 441 puntos

✓ Imagen de trayectorias generada: 'Output/trayectorias.png'
  Total de trayectorias dibujadas: 3


## 🗺️ Generación de Imagen de Trayectorias Completas

**Qué hace:** Crea una imagen estática con las trayectorias completas de todos los jugadores sobre el mapa del campo.

**Proceso:**

### **1. Reconstrucción de Trayectorias**
- Para cada jugador y cada frame:
  - Transforma su posición (cx, cy) al mapa usando la homografía **H**
  - Almacena también `cx` original para determinar el color del equipo
- Agrupa por `track_id` para obtener trayectorias completas

### **2. Dibujo de Trayectorias**
- **Líneas**: Conectan posiciones consecutivas
  - Color según equipo (azul=izquierda, rojo=derecha)
  - El color se determina por la posición promedio de cada segmento
  - Grosor: 3px

- **Punto de inicio** (primer frame):
  - Círculo blanco relleno (radio 12px)
  - Círculo de color del equipo interior (radio 8px)
  - ID del jugador encima

- **Punto final** (último frame):
  - Cuadrado relleno del color del equipo
  - Borde blanco

### **3. Leyenda**
- "EQUIPO IZQUIERDO" en azul
- "EQUIPO DERECHO" en rojo

**Salida:**
- `Output/trayectorias.png`: Imagen con trayectorias completas de todos los jugadores

In [15]:
print(f"\n{'=' * 70}")
print("VISUALIZACIÓN DEL TRACKING")
print(f"{'=' * 70}\n")

video = cv2.VideoCapture("Output/player_combined_tracking.mp4")

print("Reproduciendo video combinado...")
print("Presiona ESC para salir\n")

cv2.namedWindow("Tracking Combinado", cv2.WINDOW_NORMAL)
cv2.resizeWindow("Tracking Combinado", 1200, 900)

while True:
    ret, frame = video.read()
    if not ret:
        video.set(cv2.CAP_PROP_POS_FRAMES, 0)
        continue
    
    cv2.imshow("Tracking Combinado", frame)
    
    key = cv2.waitKey(30) & 0xFF
    if key == 27:  # ESC
        break

video.release()
cv2.destroyAllWindows()

print("\n" + "=" * 70)
print("✅ PIPELINE COMPLETADO")
print("=" * 70)
print(f"\n📊 Resumen final:")
print(f"   • Video procesado: {total_frames} frames")
print(f"   • Jugadores detectados: {len(final_ids)}")
print(f"   • IDs finales: {final_ids}")
print(f"   • Detecciones totales: {len(tracking_list)}")
print(f"   • Fusiones realizadas: {total_merges}")
print(f"   • Extras eliminados: {extras_removed}")
print(f"   • Duplicados eliminados: {duplicates_removed}")
print(f"   • Reasignaciones: {reassignments}")
print(f"   • Interpolaciones: {interpolations}")
print(f"   • Calidad: {frames_exact/len(tracking_data)*100:.1f}% frames perfectos")
print(f"\n📁 Archivos generados:")
print(f"   • Output/puntos_mapa.json")
print(f"   • Output/tracking_data.csv")
print(f"   • Output/tracking_summary.json")
print(f"   • Output/player_combined_tracking.mp4")
print(f"   • Output/trayectorias.png")
print(f"\n✓ Análisis finalizado correctamente")


VISUALIZACIÓN DEL TRACKING

Reproduciendo video combinado...
Presiona ESC para salir


✅ PIPELINE COMPLETADO

📊 Resumen final:
   • Video procesado: 490 frames
   • Jugadores detectados: 3
   • IDs finales: [2, 3, 6]
   • Detecciones totales: 1131
   • Fusiones realizadas: 24
   • Extras eliminados: 0
   • Duplicados eliminados: 655
   • Reasignaciones: 0
   • Interpolaciones: 29
   • Calidad: 0.0% frames perfectos

📁 Archivos generados:
   • Output/puntos_mapa.json
   • Output/tracking_data.csv
   • Output/tracking_summary.json
   • Output/player_combined_tracking.mp4
   • Output/trayectorias.png

✓ Análisis finalizado correctamente


## 👁️ Visualización y Reproducción del Video de Tracking

**Qué hace:** Reproduce el video de tracking generado en una ventana interactiva.

**Proceso:**
- Carga `Output/player_combined_tracking.mp4`
- Abre ventana redimensionable (1200x900)
- Reproduce en bucle continuo
- Control: presiona **ESC** para salir

**Resumen Final:**
Muestra estadísticas completas del análisis:
- Frames procesados
- Jugadores detectados (IDs finales)
- Detecciones totales
- Correcciones aplicadas (fusiones, reasignaciones, interpolaciones)
- Calidad del tracking (% frames perfectos)
- Archivos generados en `Output/`